# 📅 2026-08-29 개발 노트 : 새벽 자율 세션 — Phase 2-B 기능 완성 (Claude 단독 작업)

> 준태 취침 중 Claude가 진행. 커밋은 안 함 (자동 배포 방지) — 기상 후 검토→푸시.

## 🎯 완료된 것

- [x] **스와이프 온보딩** (`/onboarding/swipe`) — PRD Phase 2-B
- [x] **Steam 라이브러리 분석** (백엔드 API + 마이페이지 섹션) — PRD Phase 2-B
- [x] **검색 0건 폴백** — 빈 화면 대신 오늘의 추천 노출
- [x] 전체 py_compile + tsc + eslint(신규 파일) 통과

## 1. 스와이프 온보딩

**설계 결정:**
- 덱 구성: `/games/vibes`에서 5개 vibe × 3개씩 → 중복 제거 후 10개.
  단일 추천 호출보다 취향 캘리브레이션에 필요한 '다양성'이 확보됨.
- 취향 산출: 좋아요한 게임들의 `key_metrics` 평균(0~10) — 신규 API 없이
  기존 추천 응답 필드 재활용. 백엔드 변경 0줄.
- 전달: 비로그인은 sessionStorage(`hg_swipe_prefs`)로 /search에 넘겨
  도착 즉시 자동 추천 실행. 로그인 유저는 taste-preference API에도 저장
  (0~10 → 1~5 스케일 변환).
- UX: 포인터 드래그 스와이프(±80px 임계) + 버튼 + 키보드 ←/→/↓, 진행 바.

**진입점 3개:** 가입 온보딩 완료 → 자동 진입 / /search 상단 링크 / URL 직접.

**트러블:** `dragging.current`(ref)를 렌더 스타일에서 읽어 eslint
"Cannot access refs during render" → isDragging state 분리로 해결.

## 2. Steam 라이브러리 분석

**설계 결정: 무상태(stateless) v1.**
DB 컬럼 추가(마이그레이션) 대신 호출 시 Steam GetOwnedGames를 직접 조회.
무인 작업 중 손으로 migration 파일을 쓰는 리스크를 피했고, v1 트래픽에선 충분.

- `GET /api/auth/steam-library/` (Django, IsAuthenticated):
  SocialAccount(provider=steam) uid → GetOwnedGames → 플레이타임 상위 10개
  + `Game.objects.filter(app_id__in=...)`로 우리 DB 보유 여부 플래그.
- 마이페이지: 연동 유저에게만 섹션 노출. in_db 게임은 `/game/{app_id}`로
  연결("비슷한 숨은 명작 →"), 미분석 게임은 '분석 예정' 뱃지 —
  **신작 파이프라인과 자연스럽게 연결되는 훅.**
- 엣지 처리: Steam 프로필 비공개 시 빈 목록 안내, API 실패 502, 키 미설정 503.

**다음 확장:** 보유 게임 중 in_db 상위 N개를 시드로 recommend/by-game
집계 → "당신의 라이브러리 기반 추천" 섹션 (Phase 3).

## 3. 검색 0건 폴백 + 기타

- 홈 검색 0건 시: ErrorState 아래 "대신 이런 게임은 어때요? · 오늘의 테마 추천"
  그리드 노출 (이미 fetch돼 있던 aiGames 재사용, 추가 API 호출 없음).
- 참고: Next 16은 빌드에서 eslint를 안 돌리므로 기존 파일들의
  set-state-in-effect 경고는 빌드를 막지 않음 (신규 파일은 클린하게 작성).

## 📋 기상 후 진행 순서 (준태)

1. `cd frontend && npm run build` + `pytest fastapi_app/tests`
2. 로컬 확인: /onboarding/swipe 한 바퀴 → /search 자동 추천 뜨는지,
   스팀 로그인 후 마이페이지 라이브러리 섹션
3. 커밋/푸시 (`feat: swipe onboarding, steam library, search fallback`)
4. OpenAI **새 프로젝트 + 새 키** → `.env` 교체 →
   `docker compose up -d --force-recreate batch` → `batch_smoke_test`
5. 스모크 성공 시 백필: `weekly_pipeline --from 2026-03-16 --limit 500` 반복
   (실패 시 같은 명령에 `--sync`)
6. Railway: DJANGO_SECRET_KEY 확인(필수), STEAM_API_KEY 추가,
   운영 setup_oauth 1회